In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

rng = np.random.default_rng(11)
n = 2000
df = pd.DataFrame({
    "BatteryPower": rng.integers(500, 2000, n),
    "RAM_MB": rng.integers(256, 4000, n),
    "InternalMemory_GB": rng.choice([8, 16, 32, 64, 128], n),
    "ClockSpeed_GHz": rng.uniform(0.5, 3.0, n).round(2),
    "FrontCamera_MP": rng.integers(0, 20, n),
    "PrimaryCamera_MP": rng.integers(2, 20, n),
    "ScreenHeight_px": rng.integers(0, 1960, n),
    "ScreenWidth_px": rng.integers(500, 1998, n),
    "ThreeG": rng.integers(0, 2, n),
    "FourG": rng.integers(0, 2, n),
})
score = (df["RAM_MB"] * 0.8 + df["BatteryPower"] * 0.3 + df["InternalMemory_GB"] * 5
         + df["PrimaryCamera_MP"] * 20 + df["ClockSpeed_GHz"] * 100
         + df["FourG"] * 100 + rng.normal(0, 150, n))
df["PriceRange"] = pd.qcut(score, 4, labels=[0, 1, 2, 3]).astype(int)
df.to_csv("mobile_price.csv", index=False)

df = pd.read_csv("mobile_price.csv")
X = df.drop("PriceRange", axis=1)
y = df["PriceRange"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))

importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top features:\n", importance.head(5).round(4))

labels = {0: "Low cost", 1: "Medium cost", 2: "High cost", 3: "Very high cost"}
new_phone = pd.DataFrame([{
    "BatteryPower": 1500, "RAM_MB": 3000, "InternalMemory_GB": 64, "ClockSpeed_GHz": 2.2,
    "FrontCamera_MP": 8, "PrimaryCamera_MP": 12, "ScreenHeight_px": 1200,
    "ScreenWidth_px": 1000, "ThreeG": 1, "FourG": 1}])
print("\nPredicted price range for the new phone:", labels[int(model.predict(new_phone)[0])])